# Merged Visualization: Lat-Lon-Time and Lat-Height-Time with Histogram Inset

plot spectra, 
xaxis = diameter ice 
yaxis = diameter droplets

axis = colored by vapor

```
python 03-Merged-Visualization.py "cs-eriswil__20251215_205448" "200x160"  & # latest run ref+flare
python 03-Merged-Visualization.py "cs-eriswil__20251209_001346" "200x160"  &
python 03-Merged-Visualization.py "cs-eriswil__20251129_230943" "50x40"  &
python 03-Merged-Visualization.py "cs-eriswil__20251125_114053" "50x40"  &
```

In [ ]:
import sys, json
is_notebook = True if 'ipykernel' in sys.argv[0] else False

if is_notebook:
    !jupyter nbconvert --to python 03-Merged-Visualization.ipynb


In [ ]:
import os, glob
import matplotlib.pyplot as plt
import json
import pandas as pd
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)
from matplotlib.colors import LogNorm, NoNorm, ListedColormap
import matplotlib.dates as md
import multiprocessing
from matplotlib.ticker import AutoMinorLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import colormaps as pcmaps

from utilities.model_helpers import fetch_3d_data, convert_units_3d
from utilities.namelist_metadata import update_dataset_metadata
from utilities import new_fjet, new_fjet2, create_fade_cmap, set_name_tick_params
from utilities import define_bin_boundaries
import utilities.tools as tools
from utilities import load_and_prepare_holimo

from dask.diagnostics.progress import ProgressBar
from IPython.display import HTML, display
from tqdm.auto import tqdm
    

def create_new_jet3(n_colors=128, pastel_alpha=1):
    ice_colors = pcmaps.ice(np.linspace(1, 0.5, 32))
    bk_colors  = pcmaps.BkBlAqGrYeOrReViWh200(np.linspace(0.1, 0.9, 256))
    transition = np.linspace(ice_colors[-1], bk_colors[0], 16)[1:-1]
    
    # Blend with white to create pastel effect
    all_colors = np.vstack([ice_colors, transition, bk_colors])
    pastel_colors = pastel_alpha * all_colors + (1-pastel_alpha) * np.array([1, 1, 1, 1])  # 40% white blend
    
    return ListedColormap(pastel_colors)

new_jet4 = create_new_jet3(1024)


## Load Data and Setup

In [ ]:
model_data_root  = '/work/bb1262/user/schimmel/cosmo-specs-torch/cosmo-specs-runs/'
tracking_by_var = 'qi+qs'
#default
cs_runs = [
   ["cs-eriswil__20251215_205448", "200x160", 0], # 200x160
   ["cs-eriswil__20251209_001346", "200x160", 1], # 200x160
   ["cs-eriswil__20251129_230943", "50x40", 0], # 50x40
   ["cs-eriswil__20251125_114053", "50x40", 0], # 50x40)]
  ]

cs_run, ep_dom, cs_run_idx = cs_runs[1]

In [ ]:

# Load metadata
model_data_path = model_data_root + f'/RUN_ERISWILL_{ep_dom}x100/ensemble_output/{cs_run}/'
extpar_file     = model_data_root + f'/RUN_ERISWILL_{ep_dom}x100/COS_in/extPar_Eriswil_{ep_dom}.nc'
json_file       = glob.glob(model_data_path + "*.json")

print(model_data_path)
print(extpar_file)
print(json_file)
    
flist_3d = sorted(glob.glob(model_data_path + '3D_??????????????.nc'))
exp_names = [f.split('/')[-1].split('_')[-1].split('.')[0] for f in flist_3d]
with open(json_file[0], "r") as jsonfile:
    meta = json.load(jsonfile)

flare_exp_name = [exp for exp in exp_names if meta[exp]['INPUT_ORG']['sbm_par']['lflare']][cs_run_idx]
ref_exp_name = [exp for exp in exp_names if not meta[exp]['INPUT_ORG']['sbm_par']['lflare']][0]
flare_exp_nc_file = [f for f in flist_3d if flare_exp_name in f][0]
ref_exp_nc_file = [f for f in flist_3d if ref_exp_name in f][0]

# Setup plot configuration dict
cfg = {
    'flare_exp_name': flare_exp_name,
    'ref_exp_name': ref_exp_name,
    'flare_exp_nc_file': flare_exp_nc_file,
    'ref_exp_nc_file': ref_exp_nc_file,
    'resolution': '400m' if '50x40' in meta[flare_exp_name]['domain'] else '100m',
    'resolution_deg': 0.004 if '50x40' in meta[flare_exp_name]['domain'] else 0.001,
    'ydate_ini': str(meta[flare_exp_name]['INPUT_ORG']['runctl']['ydate_ini']),
    'hstart': str(meta[flare_exp_name]['INPUT_ORG']['runctl']['hstart']),
    'hstop': str(meta[flare_exp_name]['INPUT_ORG']['runctl']['hstop']),
    'dpi': 300,
    'pixel_size': (1920, 1080),
    'png_path': './',
    'poolsize': 128,
    'flare_lat': 47.07425,
    'flare_lon': 7.90522,
    'lflare': meta[flare_exp_name]['INPUT_ORG']['sbm_par']['lflare'],
    'flare_emissions': meta[flare_exp_name]['INPUT_ORG']['flare_sbm']['flare_emission'],
    'flare_dn': meta[flare_exp_name]['INPUT_ORG']['flare_sbm']['flare_dn'],
    'ishape': meta[flare_exp_name]['INPUT_ORG']['sbm_par']['ishape'],
    'iimfr': meta[flare_exp_name]['INPUT_ORG']['sbm_par']['iimfr'],
    'origin_lat': 47.070522,
    'origin_lon': 7.872991,
    'plot_xlim': (7.7671843, 7.94),
    'plot_ylim': (47.02, 47.12),
    'delta_x': 163.3876495361328,
    'delta_y': 111.1947021484375,
    'delta_t': 10.0,
    'n_lon': 186,
    'n_lat': 146,
    'n_time': 191,
    'n_alt': 20,
    'vel_lims': [-0.5, 0.5],
    'v_lims_qi': [0.0001, 1.0],
    'tick_size': 8,
    'axis_size': 9.5,
    'timer_size': 16.5,
    
}
# Load and prepare 3D data
reduced_domain = dict(latitude=slice(None,  cfg['flare_lat'] + 2.*cfg['resolution_deg']), 
                      longitude=slice(None, cfg['flare_lon'] + 2.*cfg['resolution_deg']) )
ds_3d = fetch_3d_data(flare_exp_nc_file, extpar_file, meta[flare_exp_name]['INPUT_ORG'], 
                      var_sets=['meteo', 'spec'], chunks={'time': 1, 'altitude':1})
ds_3d = update_dataset_metadata(ds_3d)
ds_3d = ds_3d.isel(altitude=slice(80, None))
ds_3d = ds_3d.sel(reduced_domain)
ds_3d = convert_units_3d(ds_3d, ds_3d["rho"])


# Load external data
data_extpar = xr.open_mfdataset(extpar_file, chunks='auto')
lat2D  = data_extpar['lat'].values[7:-7, 7:-7]
lon2D  = data_extpar['lon'].values[7:-7, 7:-7]
height = data_extpar['HSURF'].values[7:-7, 7:-7]
latitude1D  = data_extpar['lat'].mean('rlon').values[7:-7]
surf_height = data_extpar['HSURF'].values[7:-7, 7:-7]
surf_height_mean = data_extpar['HSURF'].mean('rlon').values[7:-7]

cfg.update({'n_lon':  data_extpar.rlon.size,
            'n_lat':  data_extpar.rlat.size, 
            'n_alt':  ds_3d.altitude.size,
            'n_time': ds_3d.time.size})

# Prepare lat-lon data (top view)
with ProgressBar():
    mod = ds_3d[['dz', 'qfw', 'qw', 'nf', 'nw', 't', 'qc', 'qi', 'qs', 'qv']].persist()
    
d_µm  = define_bin_boundaries() * 1.0e6 * 2.0 # radius to diameter, m to µm
mod = mod.assign_coords({'diameter_edges': xr.DataArray(d_µm, dims='diameter_edges')})
mod[tracking_by_var] = xr.where(mod['qi'] + mod['qs'] > 0, mod['qi'] + mod['qs'], np.nan)
mod[tracking_by_var].name = 'ice and snow water content'
mod[tracking_by_var].attrs['units'] = mod['qi'].attrs['units']


#     # load tobac data
#     tracks = pd.read_csv(f'{model_data_path}/{flare_exp_name}_{tracking_by_var}_tobac_track.csv').to_xarray()
#     features_mask = pd.read_csv(f'{model_data_path}/{flare_exp_name}_{tracking_by_var}_tobac_features_mask.csv').to_xarray()
#     ds_segm = xr.open_dataset(f'{model_data_path}/{flare_exp_name}_{tracking_by_var}_tobac_mask_xarray.nc')
#     ds_segm = ds_segm.sel(reduced_domain).sel( time=mod.time )
#     segmentation = ds_segm['segmentation_mask']
#     segmentation_mask = segmentation>0.0
    
#     # set values outside the plume to nan
#     cell_input = xr.where( segmentation_mask, mod, 0.0 )
    
#     return cell_input


# for tr_var in ['qi', 'qs', 'qi+qs']:
#     for cs_run, ep_dom in cs_runs:
        
#         cell_input = cs_3doutput_to_plume_nc( cs_run, ep_dom, tracking_by_var = tr_var, cs_run_idx = 1 )
        
#         # two fetch path along plume (time) is independent variable
#         plume_integrated = extract_segmented_tracks(cell_input, tracks, features_mask, **cfg) # all plume volumes ()
#         plume_maxima     = extract_segmented_tracks(cell_input, tracks, **cfg)                # maximum of segmented area of plume volumes (1 gridcell)
    
#         delayed = []
#         for key, val in plume_integrated.items():
#             delayed.append(val.to_netcdf(f'data_{cs_run}_integrated_plume_path_{tr_var}_cell{key}.nc', compute=False))
#         for key, val in plume_maxima.items():
#             delayed.append(val.to_netcdf(f'data_{cs_run}_maxima_plume_path_{tr_var}_cell{key}.nc', compute=False))
    
        
#         with ProgressBar():
#             for del_ in delayed:
#                 del_.compute()

In [ ]:

# Load HOLIMO in-situ data
holimo_data_root = os.path.abspath(os.path.join(model_data_root, os.pardir, os.pardir, 'cloudlab_data', 'holimo'))
holimo_file      = os.path.join(holimo_data_root, '2023-01-25', 'CL_20230125_1000_1140_SM058_SM060_ts1.nc')
ds_hd, lbb, cbb = load_and_prepare_holimo(holimo_file)
ds_hd = xr.where(ds_hd < 1e-7, np.nan, ds_hd)
print("\n[4] Fetching HOLIMO data ... done")

# obtain specs run identifyer

## Prepare Data for Plotting

In [ ]:
meta[flare_exp_name]['INPUT_ORG']['sbm_par']['ishape']

# PLUME PATH PLOTS

In [ ]:
from utilities import new_jet2, create_fade_cmap
from matplotlib.ticker import FuncFormatter, MultipleLocator, FixedLocator
    
def format_time_axis(ax, t0, t_end, major_interval=5, minor_interval=1):
    """Format x-axis as +MM minutes with fixed locators"""
    duration_min = (t_end - t0) / np.timedelta64(1, 'm')
    
    # Major ticks
    major_times = np.arange(0, duration_min + major_interval, major_interval)
    major_pos = [md.date2num(t0 + np.timedelta64(int(t), 'm')) for t in major_times]
    
    # Minor ticks
    minor_times = np.arange(0, duration_min + minor_interval, minor_interval)
    minor_pos = [md.date2num(t0 + np.timedelta64(int(t), 'm')) for t in minor_times]
    
    ax.xaxis.set_major_locator(FixedLocator(major_pos))
    ax.xaxis.set_minor_locator(FixedLocator(minor_pos))
    ax.set_xticklabels([f'+{int(t):02d}' for t in major_times])
    ax.tick_params(axis='x', which='major', direction='inout', length=8, width=0.9)
    ax.tick_params(axis='x', which='minor', direction='inout', length=4, width=0.4)
    ax.set_xlabel('Time (min)')

run_ini_hour = str(ds_3d.time.values[0])[11:13]
plot_time_frame = [np.datetime64(f'2023-01-25T{run_ini_hour}:25:00'), np.datetime64(f'2023-01-25T{run_ini_hour}:59:59')]

# concentrations
fig, ax = plt.subplots(nrows=2, ncols=4, figsize=(20, 8))
for cell_id in np.unique(tracks['cell'].values):

    qw_diff_i = xr.where(plume_integrated[cell_id]['qw']>0,  plume_integrated[cell_id]['qw'],  np.nan).isel(diameter=slice(None, None)).chunk({'path':4})
    qfw_diff_i = xr.where(plume_integrated[cell_id]['qfw']>0, plume_integrated[cell_id]['qfw'], np.nan).isel(diameter=slice(30,   None)).chunk({'path':4})
    qw_diff_c  = xr.where(plume_maxima[cell_id]['qw']>0,      plume_maxima[cell_id]['qw'],      np.nan).isel(diameter=slice(None, None)).chunk({'path':4})
    qfw_diff_c = xr.where(plume_maxima[cell_id]['qfw']>0,     plume_maxima[cell_id]['qfw'],     np.nan).isel(diameter=slice(30,   None)).chunk({'path':4})
    
    qw_diff_i.plot( ax=ax[0, 0], x='time', y='diameter', yscale='log',  add_colorbar=True if cell_id==1 else False, cmap=new_fjet2, alpha=0.6, extend='both', cbar_kwargs={'label': 'integrated cloud water / (kg)', "format": "{x:2.0e}"} if cell_id==1 else None ) #norm=LogNorm(vmin=1e0,  vmax=1e5), 
    qfw_diff_i.plot( ax=ax[1, 0], x='time', y='diameter', yscale='log', add_colorbar=True if cell_id==1 else False, cmap=new_fjet2, alpha=0.6, extend='both', cbar_kwargs={'label': 'integrated cloud ice / (kg)'} if cell_id==1 else None ) #norm=LogNorm(vmin=1e-3, vmax=1e1), 
    qw_diff_c.plot(  ax=ax[0, 1], x='time', y='diameter', yscale='log', add_colorbar=True if cell_id==1 else False, cmap=new_fjet2, alpha=0.6, extend='both', cbar_kwargs={'label': 'integrated cloud water / (kg)'} if cell_id==1 else None) #norm=LogNorm(vmin=1e0,  vmax=1e5), 
    qfw_diff_c.plot( ax=ax[1, 1], x='time', y='diameter', yscale='log', add_colorbar=True if cell_id==1 else False, cmap=new_fjet2, alpha=0.6, extend='both', cbar_kwargs={'label': 'integrated cloud ice / (kg)'} if cell_id==1 else None ) #norm=LogNorm(vmin=1e-3, vmax=1e1), 
    
    # rates
    qw_diff_i.differentiate(  coord='path', datetime_unit='ns').plot( ax=ax[0, 2], x='time', y='diameter', yscale='log', robust=True, add_colorbar=True if cell_id==1 else False, cmap='coolwarm', cbar_kwargs={'label': 'integrated cloud water / (kg)'} if cell_id==1 else None)
    qfw_diff_i.differentiate( coord='path', datetime_unit='ns').plot( ax=ax[1, 2], x='time', y='diameter', yscale='log', robust=True, add_colorbar=True if cell_id==1 else False, cmap='coolwarm', cbar_kwargs={'label': 'integrated cloud ice / (kg)'} if cell_id==1 else None)
    qw_diff_c.differentiate(  coord='path', datetime_unit='ns').plot( ax=ax[0, 3], x='time', y='diameter', yscale='log', robust=True, add_colorbar=True if cell_id==1 else False, cmap='coolwarm', cbar_kwargs={'label': 'integrated cloud water / (kg)'} if cell_id==1 else None)
    qfw_diff_c.differentiate( coord='path', datetime_unit='ns').plot( ax=ax[1, 3], x='time', y='diameter', yscale='log', robust=True, add_colorbar=True if cell_id==1 else False, cmap='coolwarm', cbar_kwargs={'label': 'integrated cloud ice / (kg)'} if cell_id==1 else None)


    ax[0,0].set_title(r'$\int_{V_{seg}} q_{liq}^{ijk}\,$d$V$')
    ax[0,1].set_title(r'$q_{liq}^{ijk}\,$d$V$')
    ax[1,0].set_title(r'$\int_{V_{seg}} q_{ice}^{ijk}\,$d$V$')
    ax[1,1].set_title(r'$q_{ice}^{ijk}\,$d$V$')
    
    ax[0,2].set_title(r'$\frac{\partial}{\partial\,t}\int_{V_{seg}} q_{liq}^{ijk}\,$d$V$')
    ax[0,3].set_title(r'$\frac{\partial}{\partial\,t}q_{liq}^{ijk}\,$d$V$')
    ax[1,2].set_title(r'$\frac{\partial}{\partial\,t}\int_{V_{seg}} q_{ice}^{ijk}\,$d$V$')
    ax[1,3].set_title(r'$\frac{\partial}{\partial\,t}q_{ice}^{ijk}\,$d$V$')
    t0 = plume_integrated[1].time.values[0]
    for iax in ax.flatten():
        iax.hlines(plume_integrated[1].diameter.values[30], plume_integrated[1].time.values[0], plume_integrated[1].time.values[-1], linestyle='--', color='black', alpha=0.5)
        iax.hlines(plume_integrated[1].diameter.values[50], plume_integrated[1].time.values[0], plume_integrated[1].time.values[-1], linestyle='--', color='black', alpha=0.5)
        iax.set_ylim(1, 1000)
        iax.set_xlim(*plot_time_frame)
        
        format_time_axis(iax, plume_integrated[1].time.values[0], plume_integrated[1].time.values[-1])
    
    
    for iax in ax[-1, :]:
        iax.set_xlabel('Time (min)')
    fig.subplots_adjust(hspace=0.3, wspace=0.3)
fig.savefig(f'fig_03_{flare_exp_name}_plume_evolution.png')

In [ ]:
# for i, j in zip(plume_maxima[1].path.values, plume_maxima[1].time.values):
#     print(i,j)

In [ ]:
run_ini_hour

# SINGLE SPECTRA PLOTS

In [ ]:
from utilities import new_jet, create_fade_cmap, set_name_tick_params
import matplotlib.colors as mcolors

run_ini_hour = int(cfg['ydate_ini'][-2:])
run_stop_hour = int(run_ini_hour + float(cfg['hstop']))
time_window_holimo = (np.datetime64('2023-01-25T10:30:00'),  np.datetime64('2023-01-25T10:45:00'))
time_window_specs  = (np.datetime64(f'2023-01-25T{run_ini_hour:02d}:30:00'), np.datetime64(f'2023-01-25T{run_stop_hour:02d}:00:00'))
idx_window_specs = [ np.argmin(np.abs(plume_integrated[1].time.values - time_window_specs[i])) for i in range(2)]
print(time_window_specs)
idx_window_specs

In [ ]:

# compute deltas for HOLIMO and COSMO-SPECS
dt_hol = float(np.diff(ds_hd.time.values)[0]) * 1e-9 # in seconds
dd_hol = float(np.diff(np.log10(ds_hd.diameter.values))[0])

dt_mod = float(np.diff(plume_integrated[1].time.values)[0]) * 1e-9 # in seconds
dd_mod = float(np.diff(np.log10(plume_integrated[1].diameter.values))[0])

tot_time_mod = float((time_window_specs[-1] - time_window_specs[0])/np.timedelta64(1, 's'))
tot_time_hol = float((time_window_holimo[-1]-time_window_holimo[0])/np.timedelta64(1, 's'))

plume_time = plume_integrated[1].time.values

print('dt_mod = ', dt_mod, ' sec')
print('dd_mod = ', dd_mod, '')
print()
print('dt_hol = ', dt_hol, ' sec')
print('dd_hol = ', dd_hol, '')

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
ax = ax[np.newaxis, :]

# holimo spectra integrated
spec_kw_hol = dict( x='diameter', xscale='log', yscale='log', alpha=0.7 , label='HOLIMO' ,color='royalblue', linewidth=4.75)
xr.where(ds_hd['Water_PSDnoNorm']>0, dt_hol*ds_hd['Water_PSDnoNorm'], np.nan ).sel( time=slice(*time_window_holimo) ).sum('time').plot.step(ax = ax[0, 0], **spec_kw_hol)
xr.where(ds_hd['Ice_PSDnoNorm']>0,   dt_hol*ds_hd['Ice_PSDnoNorm']*1e3,   np.nan ).sel( time=slice(*time_window_holimo) ).sum('time').plot.step(ax = ax[0, 1], **spec_kw_hol)

# cosmo-specs spectra integrated
for i in range(30, 60, 10):
    time_window_specs = [np.datetime64(f'2023-01-25T09:{i:02d}:00'), np.datetime64(f'2023-01-25T09:{i:02d}:00')+np.timedelta64(5, 'm')]
    idx_window_specs  = [ np.argmin(np.abs(plume_time - time_window_specs[i])) for i in range(2)]

    sspec_kw_cs = dict( x='diameter', xscale='log', yscale='log', alpha=0.7 ,  linewidth=2.75,  label='SPECS')
    xr.where(plume_maxima[1]['nw']>0, dt_mod*plume_maxima[1]['nw'], np.nan).isel(diameter=slice(None, None)).isel(path=slice(*idx_window_specs)).sum('path').plot.step(ax=ax[0, 0], ylim=(1e1,1e6), **sspec_kw_cs)
    xr.where(plume_maxima[1]['nf']>0, dt_mod*plume_maxima[1]['nf'], np.nan).isel(diameter=slice(30,   None)).isel(path=slice(*idx_window_specs)).sum('path').plot.step(ax=ax[0, 1], ylim=(1e1,1e6), **sspec_kw_cs)

# dd_hol*
# dd_hol*
# dd_mod*
# dd_mod*


ax[0,0].set_title(r'$n_{liq}^{ijk}\,$d$V$')
ax[0,1].set_title(r'$n_{ice}^{ijk}\,$d$V$')
# ax[0,0].set_title(r'$q_{liq}^{ijk}\,$d$V$')
# ax[0,1].set_title(r'$q_{ice}^{ijk}\,$d$V$')
set_name_tick_params(ax[0,0])
set_name_tick_params(ax[0,1])
ax[0, 0].legend()
ax[-1, -1].legend()
fig.tight_layout()
fig.savefig(f'fig_04_{flare_exp_name}_mean_spectra.png')

# TIME SERIES TEMP, MEAN QC, QI, QS, ALTITUDE,

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

# Scientific color scheme - muted, professional colors suitable for ACP/AMT/GMD
# Using distinguishable but not overly saturated colors
data = [
    ('mean_qc', 'Cloud Droplets', '#2166ac', '#b2182b'),  # dark blue / dark red
    ('mean_qi', 'Ice Crystals', '#4393c3', '#d6604d')     # medium blue / medium red-orange
]

for ax, (var, label, color_left, color_right) in zip(axes[:2], data):
    # Left axis: Growth rates
    rate_int = plume_integrated[1][var].differentiate(coord='path', datetime_unit='ns')
    rate_max = plume_maxima[1][var].differentiate(coord='path', datetime_unit='ns')
    
    ax.plot(rate_int.path, rate_int, color=color_left, linewidth=1.5, 
            label='Integrated (rate)', alpha=1.0)
    ax.plot(rate_max.path, rate_max, color=color_left, linewidth=1.5, 
            label='Maximum (rate)', linestyle='--', alpha=1.0)
    ax.set_ylabel(f'd$D_{{{var[-1]}}}$/dt (μm s$^{{-1}}$)', 
                  fontsize=10, color=color_left)
    ax.tick_params(axis='y', labelcolor=color_left)
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0, 0))
    ax.grid(alpha=0.3, linestyle=':', linewidth=0.5, color='gray')
    ax.legend(loc='upper left', framealpha=0.95, fontsize=8, frameon=True, edgecolor='gray')
    
    # Right axis: Mean diameter
    ax2 = ax.twinx()
    ax2.plot(plume_integrated[1][var].path, plume_integrated[1][var], 
             color=color_right, linewidth=1.5, alpha=1.0, label='Integrated (diameter)')
    ax2.plot(plume_maxima[1][var].path, plume_maxima[1][var], 
             color=color_right, linewidth=1.5, linestyle='--', alpha=1.0, 
             label='Maximum (diameter)')
    ax2.set_ylabel(f'$D_{{{var[-1]}}}$ (μm)', 
                   fontsize=10, color=color_right)
    ax2.tick_params(axis='y', labelcolor=color_right)
    ax2.legend(loc='upper right', framealpha=0.95, fontsize=8, frameon=True, edgecolor='gray')
    
    # Title - cleaner styling
    ax.text(0.02, 0.97, label, transform=ax.transAxes, 
            fontsize=10, weight='normal', va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', 
                     edgecolor='gray', alpha=0.9, linewidth=0.8))

# Third panel: Temperature and Altitude
ax3 = axes[2]
color_temp = '#762a83'  # purple (temperature often purple in atmos sci)
color_alt = '#1b7837'   # dark green (altitude/height)

# Left axis: Temperature
temp_int = plume_integrated[1]['temperature']
temp_max = plume_maxima[1]['temperature']

ax3.plot(temp_int.path, temp_int-273.15, color=color_temp, linewidth=1.5, 
         label='Integrated', alpha=1.0)
ax3.plot(temp_max.path, temp_max-273.15, color=color_temp, linewidth=1.5, 
         label='Maximum', linestyle='--', alpha=1.0)

ax3.set_ylim(-6, -3)
ax3.set_ylabel('Temperature (°C)', fontsize=10, color=color_temp)
ax3.tick_params(axis='y', labelcolor=color_temp)
ax3.grid(alpha=0.3, linestyle=':', linewidth=0.5, color='gray')
ax3.legend(loc='upper left', framealpha=0.95, fontsize=8, frameon=True, edgecolor='gray')

# Right axis: Altitude
ax3_2 = ax3.twinx()
alt_int = plume_integrated[1]['altitude']
alt_max = plume_maxima[1]['altitude']

ax3_2.plot(alt_int.path, alt_int, color=color_alt, linewidth=1.5, 
           alpha=1.0, label='Integrated')
ax3_2.plot(alt_max.path, alt_max, color=color_alt, linewidth=1.5, 
           linestyle='--', alpha=1.0, label='Maximum')

ax3_2.set_ylabel('Altitude (m)', fontsize=10, color=color_alt)
ax3_2.tick_params(axis='y', labelcolor=color_alt)
ax3_2.legend(loc='upper right', framealpha=0.95, fontsize=8, frameon=True, edgecolor='gray')

ax3.text(0.02, 0.97, 'Atmospheric Profile', transform=ax3.transAxes, 
         fontsize=10, weight='normal', va='top',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', 
                  edgecolor='gray', alpha=0.9, linewidth=0.8))

axes[2].set_xlabel('Path Index', fontsize=10)
fig.tight_layout()
fig.savefig(f'fig_05_{flare_exp_name}_path-series-c-cbar-temp-height.png')

In [ ]:
# def compute_integrated_water_contents(ds_3d):
#     """
#     Compute domain-integrated water contents over time.
#     Returns time series of IWC, LWC, SWC, TWC in kg.
#     """
#     # Select relevant altitude range
#     dss = ds_3d[['qc', 'qi', 'qs', 'qv', 'dz']]
#     wc = {
#         'IWC': (dss['qi'] * dss['dz'] * cfg['delta_x'] * cfg['delta_y']).sum(['altitude', 'longitude', 'latitude']),
#         'LWC': (dss['qc'] * dss['dz'] * cfg['delta_x'] * cfg['delta_y']).sum(['altitude', 'longitude', 'latitude']),
#         'SWC': (dss['qs'] * dss['dz'] * cfg['delta_x'] * cfg['delta_y']).sum(['altitude', 'longitude', 'latitude']),
#         'VC':  (dss['qv'] * dss['dz'] * cfg['delta_x'] * cfg['delta_y']).sum(['altitude', 'longitude', 'latitude']),
#     }
#     wc.update({
#         'TLWC': wc['IWC'] + wc['LWC'],
#         'TIWC': wc['IWC'] + wc['SWC']
#     })
#     return xr.Dataset(wc)

# # Compute water contents
# water_contents = compute_integrated_water_contents(mod)


# print("Water content time series computed")
# print(f"{'VAR':>5s}:  {'min':>15s},  {'max':>15s}")
# print(f"{'-'*50}")
# for key, val in water_contents.items():
#     print(f"{key:>5s}:  {val.min().values:20.4f},  {val.max().values:20.4f}")


In [ ]:
# Prepare lat-lon data (top view)
SMALL = 1e-7
_tmp = (mod[['qi', 'qs']] * mod['dz']).sum('altitude')
_tmp = xr.where(_tmp < SMALL, np.nan, _tmp)
IWC_lat_lon = (_tmp['qi'] + _tmp['qs']).values
print(f"IWC_lat_lon (lat-lon) shape: {IWC_lat_lon.shape}")

# Prepare lat-height data (vertical view)
_tmp = mod[['qi', 'qs']].sel( latitude=slice(None, cfg['flare_lat'] + 2.*cfg['resolution_deg']), 
                              longitude=slice(None, cfg['flare_lon'] + 2.*cfg['resolution_deg']) )
_tmp = xr.where(_tmp < SMALL, np.nan, _tmp)
_tmp = (_tmp * cfg['delta_x']).sum('longitude')

altitude1D = _tmp.altitude.values
IWC_lat_alt = (_tmp['qi'] + _tmp['qs']).values
print(f"IWC_lat_lon (lat-height) shape: {IWC_lat_alt.shape}")

tracking_by_var = 'qi+qs'
#tracks_csv    = tools.load_tracking_csv(f'{model_data_path}/{flare_exp_name}_qi+qs_tobac_track.csv')
tracks_csv_qi = tools.load_tracking_csv(f'{model_data_path}/{flare_exp_name}_qi_tobac_track.csv')
tracks_csv_qs = tools.load_tracking_csv(f'{model_data_path}/{flare_exp_name}_qs_tobac_track.csv')
#cell_tracks    = [ group[1][pd.to_datetime(group[1]['time']) >= plot_time_frame[0]][::6] for group in tracks_csv.groupby("cell") ]
cell_tracks_qi = [ group[1][pd.to_datetime(group[1]['time']) >= plot_time_frame[0]][::6] for group in tracks_csv_qi.groupby("cell") ]
cell_tracks_qs = [ group[1][pd.to_datetime(group[1]['time']) >= plot_time_frame[0]][::6] for group in tracks_csv_qs.groupby("cell") ]

# # precompute histos
# masked_qv = np.where(segmentation_mask > 0, mod['qv'], np.nan)
# masked_qc = np.where(segmentation_mask > 0, mod['qc'], np.nan)

# nbins = 40
# xbins = np.linspace( np.nanmin(masked_qv).round()-1, np.ceil(np.nanmax(masked_qv)), nbins )
# ybins = np.linspace( np.nanmin(masked_qc).round(), np.ceil(np.nanmax(masked_qc)), nbins )
# bins = [xbins, ybins]
# print(bins)

# histograms = update_histogram( masked_qv, masked_qc, bins)



## Define Merged Plotting Functions

In [ ]:
def set_name_tick_params(ax):
    ax.tick_params(which='both', direction='inout', top=True, right=True, bottom=True, left=True)
    ax.minorticks_on()
    ax.tick_params(which='major', length=5)
    ax.tick_params(which='minor', length=3)
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')
    ax.grid(True, which='major', linestyle='--', linewidth='0.11', color='black', alpha=0.5, zorder=99.1)
    ax.grid(True, which='minor', linestyle=':', linewidth='0.075', color='black', alpha=0.25, zorder=99.1)
    ax.set_axisbelow(False)

def setup_axes_grid(cfg):
    """
    Initialize merged figure with:
    - Top left: Lat-lon view
    - Bottom left: Lat-height view
    - Right: Time series
    - Inset in lat-height: Histogram
    """
    plt.style.use('seaborn-v0_8-paper')
    
    # Create figure with custom layout
    fig = plt.figure(figsize=(12, 8), constrained_layout=True)
    
    # Create grid: 2 rows, 2 columns
    # Left column: lat-lon (top) and lat-height (bottom)
    # Right column: time series (spans both rows)
    gs = fig.add_gridspec(2, 2, 
                          width_ratios  = [2.0, 1.0], 
                          height_ratios = [1.0, 1.2], 
                          hspace        = 0.08, 
                          wspace        = 0.1)
    
    ax_latlon    = fig.add_subplot(gs[0, 0]) # Top left
    ax_latheight = fig.add_subplot(gs[1, 0]) # Bottom left
    ax_ts_top    = fig.add_subplot(gs[0, 1]) # Right top
    ax_ts_bottom = fig.add_subplot(gs[1, 1]) # Right bottom
    
    # Create inset axis for histogram in top right of lat-height plot
    ax_hist      = inset_axes(ax_latheight, width="30%", height="40%", loc='upper right', 
                              bbox_to_anchor=(-0.05, -0.05, 1.0, 1.0), bbox_transform=ax_latheight.transAxes)
    
    # Configure lat-lon axis
    set_name_tick_params(ax_latlon)
    ax_latlon.set_xlim(cfg['plot_xlim'])
    ax_latlon.set_ylim(cfg['plot_ylim'])
    ax_latlon.tick_params(axis='both', which='major', labelsize=cfg['tick_size'])
    ax_latlon.set_xlabel('longitude / (°)', fontsize=cfg['axis_size'])
    ax_latlon.set_ylabel('latitude / (°)', fontsize=cfg['axis_size'])
    
    # Configure lat-height axis
    set_name_tick_params(ax_latheight)
    ax_latheight.set_ylim((600, 1400))
    ax_latheight.set_xlabel('latitude / (°)', fontsize=cfg['axis_size'])
    ax_latheight.set_ylabel('height / (m)', fontsize=cfg['axis_size'])
    
    # Configure time series axis
    for ax in [ax_ts_top, ax_ts_bottom]:
        set_name_tick_params(ax)
        ax.set_xlabel('time / (UTC)', fontsize=cfg['axis_size'], color='black')
        ax.set_axisbelow(True)
        ax.xaxis.set_major_formatter(md.DateFormatter('%H:%M'))
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.set_yscale('log')
    
    # ax_ts.set_ylabel('ice mass / (kg)', fontsize=cfg['axis_size'], color='black')
    # ax_ts.ticklabel_format(axis='y', style='scientific', scilimits=(-3, +3))
    # ax_ts.tick_params(axis='both', which='major', direction='inout', top=True, right=True, labelsize=cfg['tick_size'])
    # ax_ts.tick_params(axis='both', which='minor', direction='inout', top=True, right=True, labelsize=cfg['tick_size'])
    # ax_ts.grid(which='both', linestyle='--', alpha=0.15, linewidth=0.5, color='black')
    # ax_ts.grid(which='minor', linestyle='--', alpha=0.05, linewidth=0.25, color='black')

    # Configure histo axis
    ax_hist.tick_params(axis='both', which='major', labelsize=cfg['tick_size']-2)
    # ax_hist.set_xlabel('diameter', fontsize=cfg['tick_size']-1)
    # ax_hist.set_ylabel('cumulative\ncount', fontsize=cfg['tick_size']-1)
    ax_hist.patch.set_alpha(0.8)
    ax_hist.tick_params(which='both', direction='in', top=True, right=True, bottom=True, left=True)
    ax_hist.minorticks_on()
    ax_hist.tick_params(which='major', length=5)
    ax_hist.tick_params(which='minor', length=3)
    ax_hist.xaxis.set_ticks_position('both')
    ax_hist.yaxis.set_ticks_position('both')
    ax_hist.xaxis.tick_top()
    ax_hist.yaxis.tick_right()
    
    ax_hist.xaxis.set_label_position("top")
    ax_hist.yaxis.set_label_position("right")
    ax_hist.grid(True, which='major', linestyle='--', linewidth='0.11', color='black', alpha=0.5, zorder=99.1)
    ax_hist.grid(True, which='minor', linestyle=':', linewidth='0.075', color='black', alpha=0.25, zorder=99.1)
    ax_hist.set_xscale('log')
    ax_hist.set_yscale('log')
    
    return fig, ax_latlon, ax_latheight, ax_ts_top, ax_ts_bottom, ax_hist


def add_annotations(ax, cfg):
    """Add scatter points, ruler, and seeding paths to lat-lon plot."""
    # Scatter points
    ax.scatter(cfg['origin_lon'], cfg['origin_lat'], s=70, marker='x', color='red', zorder=99)
    ax.scatter(cfg['flare_lon'],  cfg['flare_lat'],  s=50, marker='o', facecolor='none', edgecolor='white', linewidth=2.5, zorder=98)
    ax.scatter(cfg['flare_lon'],  cfg['flare_lat'],  s=50, marker='o', facecolor='none', edgecolor='red', linewidth=1.0, zorder=99)
    
    # Ruler
    tools.add_ruler(ax, 47.05, 7.804, cfg['flare_lat'], cfg['flare_lon'])
    
    # # Seeding paths
    # seeding_coords = {
    #     'royalblue': ([7.90476, 7.90568], [47.07602, 47.07248]),
    #     'orange':    ([7.89828, 7.89919], [47.07526, 47.07172]),
    #     'green':     ([7.91125, 7.91216], [47.07676, 47.07322]),
    # }
    
    # for color, (lon, lat) in seeding_coords.items():
    #     dx, dy = np.diff(lon)[0], np.diff(lat)[0]
    #     extended_lon = [lon[0] - dx, lon[1] + dx]
    #     extended_lat = [lat[0] - dy, lat[1] + dy]
        
    #     ax.plot(lon, lat, color='black', linewidth=0.9)
    #     ax.plot(extended_lon, extended_lat, color='black', alpha=0.4, linewidth=0.6)
    #     ax.scatter(lon, lat, color='black', marker='.', s=15)


# def plot_timeseries_dual_axis(ax_ts, IWC, SWC, TIWC, current_time, cfg):
#     """Plot time series of water contents with vertical line at current time."""
#     ax_ts.clear()
    
#     time = water_contents['IWC'].time.values
    
#     # Plot water content time series
#     colors_left = {'IWC': '#2E86AB', 'SWC': '#F18F01', 'TIWC': '#000000'}
#     lines = []
#     labels = ['IWC', 'SWC', 'TIWC']
#     for key in labels:
#         line, = ax_ts.plot(time, water_contents[key].values*1e-3,
#                           color=colors_left[key],
#                           linewidth=2.5 if key == 'TIWC' else 1.5,
#                           alpha=0.8 if key == 'TIWC' else 0.9,
#                           label=key)
#         lines.append(line)
    
#     # Add vertical line for current time
#     ax_ts.axvline(current_time, color='red', linestyle='--', linewidth=2, alpha=0.75, zorder=10)
#     ax_ts.set_xlim(time[0], time[-1])
#     ax_ts.set_ylim(1e-2, 1e4)
#     # ax_ts.set_xlim(time[0], time[-1])
#     # ax_ts.set_ylim(1e-2, 1e4)
#     # ax_ts.set_xlabel('time / (UTC)', fontsize=cfg['axis_size'], color='black')
#     # ax_ts.set_ylabel('ice mass / (kg)', fontsize=cfg['axis_size'], color='black')
#     # ax_ts.set_axisbelow(True)
#     # ax_ts.ticklabel_format(axis='y', style='scientific', scilimits=(-3, +3))
#     # ax_ts.xaxis.set_major_formatter(md.DateFormatter('%H:%M'))
#     # ax_ts.xaxis.set_minor_locator(AutoMinorLocator())
#     # ax_ts.yaxis.set_minor_locator(AutoMinorLocator())
#     # ax_ts.tick_params(axis='both', which='major', direction='inout', top=True, right=True, labelsize=cfg['tick_size'])
#     # ax_ts.tick_params(axis='both', which='minor', direction='inout', top=True, right=True, labelsize=cfg['tick_size'])
#     # ax_ts.grid(which='both', linestyle='--', alpha=0.15, linewidth=0.5, color='black')
#     # ax_ts.grid(which='minor', linestyle='--', alpha=0.05, linewidth=0.25, color='black')
#     # ax_ts.legend(lines, labels, loc='upper left', ncol=1, fontsize=cfg['tick_size'], framealpha=0.9, edgecolor='none')
#     # ax_ts.set_yscale('log')
#     return lines

# def update_histogram(masked_v1, masked_v2, bins):
#     histo_list = []

#     for iframe in range(mod.time.size):
#         masked_v1_t_flat = masked_v1[iframe, :, :].flatten()
#         masked_v2_t_flat = masked_v2[iframe, :, :].flatten()
        
#         # masked_v1_t_flat = masked_v1_t_flat[masked_v1_t_flat > 0]
#         # masked_v2_t_flat = masked_v2_t_flat[masked_v2_t_flat > 0]
        
#         if (len(masked_v1_t_flat) > 0) and (len(masked_v2_t_flat) > 0):
#             hist_counts, xbin_edges, ybin_edges = np.histogram2d(masked_v1_t_flat, masked_v2_t_flat, bins=bins)
#             print(xbin_edges[0], ybin_edges[0] )
#             print(xbin_edges[-1], ybin_edges[-1] )
#         else:
#             nbins = len(bins[0]) - 1
#             hist_counts = np.zeros((nbins,nbins))
        
#         histo_list.append(hist_counts)
        
#     return np.array(histo_list)


## Initialize Merged Plot

In [ ]:
import utilities
import importlib
importlib.reload(utilities)

In [ ]:
# from utilities import new_fjet3

# ########################
# # Initialize merged plot
# fig, ax_latlon, ax_latheight, ax_ts_top, ax_ts_bottom, ax_hist = setup_axes_grid(cfg)


# ########################
# # Create base layers for lat-lon plot
# pm_surf_latlon = ax_latlon.pcolormesh(lon2D, lat2D, height, cmap='terrain', vmin=300, alpha=0.6)
# pm_qi_latlon = ax_latlon.pcolormesh(mod.longitude, mod.latitude, IWC_lat_lon[100, :, :], cmap=new_fjet2, alpha=0.7, norm=LogNorm(*cfg['v_lims_qi']), zorder=2, label='IWP')
# pmesh_latlon = [pm_qi_latlon, pm_surf_latlon]
# cell_plots = []

# # Add annotations to lat-lon plot
# add_annotations(ax_latlon, cfg)

# # Add colorbar for lat-lon plot
# cb_qi_latlon = fig.colorbar(pm_qi_latlon, ax=ax_latlon, extend='both', shrink=0.7, aspect=15, pad=0.02)
# cb_qi_latlon.ax.text(0.52, 4e-1 * cfg['v_lims_qi'][0], 'IWP\n(g/m²)', ha='center', va='top', fontsize=cfg['axis_size']-3)


# ########################
# # Create base layers for lat-height plot
# pm_surf_latheight = ax_latheight.plot(latitude1D, surf_height_mean, color='black', alpha=0.9, linewidth=2)
# pml_surf_latheight = ax_latheight.plot(latitude1D, surf_height, color='black', alpha=0.035 if '100m' in cfg['resolution'] else 0.075)
# pm_qi_latheight = ax_latheight.pcolormesh(latitude1D[:IWC_lat_alt.shape[-1]], altitude1D, IWC_lat_alt[100, :, :], cmap=new_fjet2, alpha=0.7, norm=LogNorm(*cfg['v_lims_qi']), zorder=2, label='IWP')
# pmesh_latheight = [pm_qi_latheight, pm_surf_latheight]

# # Add colorbar for lat-height plot
# cb_qi_latheight = fig.colorbar(pm_qi_latheight, ax=ax_latheight, pad=0.025, extend='both', shrink=0.8, aspect=20, label='IWP (g/m²)')
# cb_qi_latheight.ax.tick_params(labelsize=cfg['tick_size'])


# ########################
# # Plot initial time series
# # plot_timeseries_dual_axis(ax_ts_top, water_contents, mod.time.values[0], cfg)
# # ax.legend(lines, labels, loc='upper left', ncol=1, fontsize=cfg['tick_size'], framealpha=0.9, edgecolor='none')

# ########################
# # plot initial histo
# from xhistogram.xarray import histogram as xhist

# def update_histogram(mo1, mo2, bins, weights=None):
#     histo_list = {}
#     for iframe in range(mo1.time.size):
#         da1 = mo1.isel(time=iframe)
#         da2 = mo2.isel(time=iframe)
#         histo_list[iframe]  = xhist(da1, da2, bins=bins, weights=weights)
#     return histo_list

# nbins=100
# xbins = np.logspace( 0.2, 0.7, num=nbins , base=10)
# ybins = np.logspace( -5, 0, num=nbins , base=10)
# bins = [xbins, ybins]
# # h  = update_histogram(da['qv'], da['qc'], bins)
# hw = update_histogram(mod['qv'], mod['qc'], bins, weights=mod['qi']+mod['qs'])

# print('x   ', bins[0][0], bins[0][-1])
# print('y   ', bins[1][0], bins[1][-1])


# i=100
# pmesh_hist = ax_hist.pcolormesh(bins[0], bins[1], xr.where(hw[i]>0, hw[i], np.nan), cmap=new_fjet2, alpha=0.7, norm=LogNorm(vmin=1e-8, vmax=1e0),)
# # xr.where(h[i]>0, h[i], np.nan).plot(ax=ax[j, 0], xscale='log', yscale='log', norm=LogNorm(vmin=1e0, vmax=1e2), cmap=new_jet2) #LogNorm(vmin=1e1, vmax=5e4)
# # xr.where(hw[i]>0, hw[i], np.nan).plot(ax=ax[j, 1], xscale='log', yscale='log', norm=LogNorm(vmin=1e-8, vmax=1e0), cmap=new_jet2) 
# # ax_hist.set_xlabel('diameter', fontsize=cfg['tick_size']-1)
# # ax_hist.set_ylabel('cumulative\ncount', fontsize=cfg['tick_size']-1, rotation=90, labelpad=2)
# ax_hist.set_xlim(bins[0][0], bins[0][-1])
# ax_hist.set_ylim(bins[1][0], bins[1][-1])
# ax_hist.set_xticks([0.2, 1, 7.0])
# ax_hist.set_xticklabels([f'{xtick:.1f}' for xtick in ax_hist.get_xticks()])


# print("Merged plot initialized")

# plt.show()

# Generate Animation Frames

## Define update frame function

In [ ]:
def update_merged_frame(iframe, 
                        fig, ax_latlon, ax_latheight, ax_ts_top, ax_ts_bottom, ax_hist, 
                        pmesh_latlon, pmesh_latheight, cell_plots, pmesh_hist, cfg, 
                        time_values, IWC_lat_lon, IWC_lat_alt, water_contents, 
                        cell_tracks_1, cell_tracks_2, histo_list  ):
    """
    Update all subplots for animation frame.
    """
    
    from matplotlib.colors import LogNorm
    # Clear previous timer text
    for txt in fig.texts:
        if 'UTC' in txt.get_text():
            txt.set_visible(False)
    
    # Update lat-lon plot
    pmesh_latlon[0].set_array(IWC_lat_lon[iframe, :, :].flatten())
    
    # Update tracks on lat-lon plot
    if cell_tracks_1 or cell_tracks_2:
        for plot in cell_plots:
            plot.remove()
        cell_plots.clear()
    
    if cell_tracks_1 is not None:
        for cell_track in cell_tracks_1:
            track = cell_track[pd.to_datetime(cell_track['time']) <= time_values[iframe]]
            if not track.empty:
                cell_plots.append(
                    ax_latlon.scatter(track.longitude.values, track.latitude.values, c='#2E86AB', s=30, alpha=0.95, marker='o', zorder=99) )
    
    if cell_tracks_2 is not None:
        for cell_track in cell_tracks_2:
            track = cell_track[pd.to_datetime(cell_track['time']) <= time_values[iframe]]
            if not track.empty:
                cell_plots.append(
                    ax_latlon.scatter(track.longitude.values, track.latitude.values, c='#F18F01', s=30, alpha=0.95, marker='+', zorder=99) )
    
    pmesh_latheight[0].set_array(IWC_lat_alt[iframe, :, :].flatten())           # Update lat-height plot
    pmesh_hist.set_array(xr.where(hw[iframe]>0, hw[iframe], np.nan).values.flatten())                    # Update histogram
    plot_timeseries_dual_axis(ax_ts, water_contents, time_values[iframe], cfg)  # Update time series plot
    
    # Add timer
    timer_str = np.datetime_as_string(time_values[iframe], unit='s').replace('T', '  ')
    fig.text(0.98, 0.98, f'{timer_str[-8:]} UTC', ha='right', va='top', fontweight='semibold', fontsize=cfg['timer_size'])
    
    out_file = f"3D_merged_{iframe:03d}_{cfg['resolution']}.png"
    out_dir = os.path.join(cfg['png_path'], 'QL-Merged')
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, out_file), dpi=cfg['dpi'], bbox_inches='tight')

In [ ]:
# # Generate frames in parallel
# from tqdm.auto import tqdm

# poolsize = 128
# model_time = mod.time.values

# def render_frame(iframe):
#     update_merged_frame(iframe, 
#                         fig, ax_latlon, ax_latheight, ax_ts_top, ax_ts_bottom, ax_hist,
#                         pmesh_latlon, pmesh_latheight, cell_plots, pmesh_hist,
#                         cfg, model_time, IWC_lat_lon, IWC_lat_alt, water_contents,
#                         cell_tracks_qi, cell_tracks_qs, hw)


# # Clean output directory
# frame_dest = os.path.join(cfg['png_path'], 'pngs')
# os.makedirs(frame_dest, exist_ok=True)
# # for f in glob.glob(os.path.join(cfg['png_path'], 'pngs/*.png')):
# #     os.remove(f)
# print(f"Output directory: {frame_dest}/pngs")

# nframes = mod.time.size

# with multiprocessing.Pool( poolsize ) as pool:
#     pool.map( render_frame, range(nframes) )

# # for i in tqdm(range(nframes)):
# #     render_frame(i)
    
# print('All frames created')



In [ ]:
# # Create video and gif
# input_pattern = os.path.join(cfg['png_path'], f"pngs/3D_merged_%03d_{cfg['resolution']}.png")
# output_mp4 = os.path.join(cfg['png_path'], f"5-eriswil_merged_{flare_exp_name}_{cfg['resolution']}.mp4")
# output_gif = output_mp4.replace('.mp4', '.gif')

# tools.convert_to_video(input_pattern, output_mp4, resolution="1920:1080", loop_count=2, framerate=15)
# tools.convert_to_gif(input_pattern, output_gif, scale_factor=0.5, fps=10)

# print(f"Created: {output_mp4}")
# print(f"Created: {output_gif}")


In [ ]:
from xhistogram.xarray import histogram as xhist
from utilities import new_jet3

cmap_rf = ListedColormap(pcmaps.rainforest(np.linspace(0.0, 0.85, 256)))
cmap_sv = ListedColormap(pcmaps.savanna(np.linspace(0.0, 0.85, 256)))
cmap_rf_fade = create_fade_cmap(cmap_rf, 16)
cmap_sv_fade = create_fade_cmap(cmap_sv, 16)


def setup_histo_fig(figsize=(5, 8), nrows=1, grid=True, labelsize=None, sharex=True):
    fig, axes = plt.subplots(nrows=nrows, figsize=figsize, constrained_layout=True, sharex=sharex)
    # Configure time series axis
    if nrows > 1:
        
        axe = axes.flatten() 
    else:
        axe = [ axes ]
    
    for ax in axe:
        set_name_tick_params(ax)
        ax.tick_params(axis='both', which='major', labelsize=cfg['tick_size']-2 if labelsize is None else labelsize-1)
        ax.patch.set_alpha(0.8)
        ax.tick_params(which='both', direction='in', top=True, right=True, bottom=True, left=True)
        ax.minorticks_on()
        ax.tick_params(which='major', length=5)
        ax.tick_params(which='minor', length=3)
        ax.xaxis.set_ticks_position('both')
        ax.yaxis.set_ticks_position('both')
        # ax.yaxis.tick_right()
        # ax.yaxis.set_label_position("right")
        if grid:
            ax.grid(True, which='major', linestyle='--', linewidth='0.11', color='black', alpha=0.75, zorder=99.1)
            ax.grid(True, which='minor', linestyle=':', linewidth='0.075', color='black', alpha=0.5, zorder=99.1)
    return fig, axes



def plot_fig8_histo(hist, nrows=1, figsize=(6, 6), robust=False, clabel="histogram abs. count", 
                    ylim=None, xlim=None,norm=None):
    

    fig, axes = setup_histo_fig(figsize=figsize, nrows=nrows, grid=False, labelsize=10)
    iax = axes.flat[0] if nrows > 1 else axes

    xr.where(hist>0,  hist,  np.nan).plot(ax=iax, 
                                          robust=robust, 
                                          norm=norm,
                                          alpha=0.9, 
                                          cmap=new_jet4,
                                          cbar_kwargs={"orientation": "vertical", 
                                                       "location": "right", 
                                                       "extend": "both", 
                                                       "shrink": 0.9, "pad":0.025,
                                                       'label': clabel, })

    # for iax in axes:
    iax.set_yscale('log')
    iax.set_xscale('log')
    if ylim is not None:
        iax.set_ylim(*ylim)
    if xlim is not None:
        iax.set_xlim(*xlim)
    # iax.xaxis.tick_top()
    iax.set_ylabel(f"water vapor / ({mod['qv'].attrs['units']})", fontsize=10)
    iax.set_xlabel(f"liquid water / ({mod['qc'].attrs['units']})", fontsize=10)
    return fig, axes


In [ ]:
#plot spectra, 
#xaxis = diameter ice 
#yaxis = diameter droplets
#zaxis = colored by vapor


d_z = mod.dz.values
d_Vol  = cfg['delta_x'] * cfg['delta_y'] * d_z
mo_Vol_dict = []
# mo_Vol_max_list = []
# for ibin in range(0, mod.diameter.size):
#     delta_D = mod.diameter_edges[ibin+1] - mod.diameter_edges[ibin]
#     mo_Vol = mod[['qfw', 'qw']].isel(diameter=ibin) 
#     mo_Vol = mo_Vol # * d_Vol * delta_D
#     mo_Vol_dict.append( mo_Vol.expand_dims('diameter', axis=-1) )


# mo_Vol_spec = xr.concat(mo_Vol_dict, dim='diameter', )
mo_Vol_spec = mod[['nf', 'nw',
                   'qfw', 'qw']]

with ProgressBar():
    mo_Vol_spec = mo_Vol_spec.persist()
    mo_Vol_spec = mo_Vol_spec.assign_coords({'diameter_edges': mod.diameter_edges})
    mo_Vol_spec = xr.where(mo_Vol_spec>0, mo_Vol_spec, np.nan)

In [ ]:
mo_Vol_spec

In [ ]:
from tqdm.auto import tqdm

xlim_logq = [np.floor(np.log10(np.nanmin(mo_Vol_spec['qfw'].values))),  np.floor(np.log10(np.nanmax(mo_Vol_spec['qfw'].values)))]
ylim_logq = [np.floor(np.log10(np.nanmin(mo_Vol_spec['qfw'].values))),  np.floor(np.log10(np.nanmax(mo_Vol_spec['qw'].values)))]
xlim_logn = [np.floor(np.log10(np.nanmin(mo_Vol_spec['nf'].values))),   np.floor(np.log10(np.nanmax(mo_Vol_spec['nf'].values)))]
ylim_logn = [np.floor(np.log10(np.nanmin(mo_Vol_spec['nw'].values))),   np.floor(np.log10(np.nanmax(mo_Vol_spec['nw'].values)))]
# xlim_log = [np.floor(np.log10(np.nanmin(mo_Vol_spec.diameter_edges.values))), np.floor(np.log10(np.nanmax(mo_Vol_spec.diameter_edges.values)))]
# ylim_log = xlim_log

# xlim_log = [-8, 7]
# ylim_log = [-15, 5]

print(xlim_logq,xlim_logn )
print(ylim_logq,ylim_logn  )

nbins = 64
xbinsq = np.logspace( *xlim_logq, num=nbins , base=10)
ybinsq = np.logspace( *ylim_logq, num=nbins , base=10)
xbinsn = np.logspace( *xlim_logn, num=nbins , base=10)
ybinsn = np.logspace( *ylim_logn, num=nbins , base=10)
print(xbinsq[0], xbinsq[-1])
print(ybinsq[0], ybinsq[-1])

ntime, nDbins = mo_Vol_spec.time.size, mo_Vol_spec.diameter.size
hspec_time_list = []
for itime in tqdm(range(ntime)):
    hspec_bin_list = []
    _m = mo_Vol_spec.isel(time=itime)
    for ibin in range(nDbins):
        __m = _m.isel(diameter=ibin)
        __hq = xhist( __m['qfw'], __m['qw'], bins=[xbinsq, ybinsq] ) 
        __hn = xhist( __m['nf'],  __m['nw'], bins=[xbinsn, ybinsn] ) 
        hspec_bin_list.append(xr.merge([__hq, __hn]).expand_dims('diameter'))
    _h = xr.concat(hspec_bin_list, dim='diameter').expand_dims('time')
    hspec_time_list.append( _h )
ds_hspec_time_q_n = xr.concat(hspec_time_list, dim='time')

In [ ]:
ds_plot = ds_hspec_time_q_n['histogram_qfw_qw'].sum('qw_bin').assign_coords({'diameter': mo_Vol_spec.diameter})
ds_plot = xr.where(ds_plot>0, ds_plot, np.nan)

In [ ]:
ds_plot

In [ ]:

ds_plot.isel(time=slice(None,None,10)).plot(  
    x='diameter', #y='nf_bin',
    col='time',
            yscale='log', 
           xscale='log',
           robust=True, 
            col_wrap=5,
           # norm=LogNorm(vmin=vlim[0], vmax=vlim[1])  ,
           norm=LogNorm(),
           alpha=0.9, 
           # xlim=xlim, 
           # ylim=ylim,
           cmap=new_jet4,
           cbar_kwargs={"orientation": "vertical", 
                       "location": "right", 
                       "extend": "both", 
                       "shrink": 0.9, "pad":0.025,
                       'label': "FoO liq/frozen water \nspectra integral", })

In [ ]:

print(len(hspec_time_list))
ds_hspec_time_q_n = xr.concat(hspec_time_list, dim='time', )
ds_hspec_q_n      = ds_hspec_time_q_n.sum('time')
ds_hspec_q_n      = xr.where(ds_hspec_q_n>0, ds_hspec_q_n, np.nan)

with ProgressBar():
    ds_hspec_q_n = ds_hspec_q_n.persist()

In [ ]:
print(ds_hspec_q_n.data_vars)
print(ds_hspec_q_n.sizes)

In [ ]:
ds_hspec = ds_hspec_q_n['histogram_qfw_qw'].isel(diameter=40)
xbins = xbinsq
ybins = ybinsq
    
# print(np.nanmin(ds_hspec), np.nanmax(ds_hspec))

xlim = (xbins.min(), xbins.max())
ylim = (ybins.min(), ybins.max())
vlim = (10**np.ceil(np.log10(ds_hspec.min().values)), 
        10**(np.floor(np.log10(ds_hspec.max().values))+1))

cb_kw = {"cbar_kwargs": {"orientation": "vertical", "location": "right", "extend": "both", "shrink": 0.9, "pad":0.1 , 'label': "histogram / (# per bin)"}}

fig, ax = setup_histo_fig(figsize=(6,5), nrows=1, grid=False, labelsize=10)

ds_hspec.plot(ax=ax, 
           yscale='log', 
           xscale='log',
           robust=True, 
           # norm=LogNorm(vmin=vlim[0], vmax=vlim[1])  ,
           norm=LogNorm(),
           alpha=0.9, 
           # xlim=xlim, 
           # ylim=ylim,
           cmap=new_jet4,
           cbar_kwargs={"orientation": "vertical", 
                       "location": "right", 
                       "extend": "both", 
                       "shrink": 0.9, "pad":0.025,
                       'label': "FoO liq/frozen water \nspectra integral", })
# ax.plot([1e-18, 1e7], [1e-18, 1e7], '--')
# ax.set_xlim(1e-18, 1e7)
# ax.set_ylim(1e-18, 1e7)
ax.set_ylabel(f"liquid water spectra / ({mod['qw'].attrs['units']})", fontsize=10)
ax.set_xlabel(f"frozen water spectra / ({mod['qfw'].attrs['units']})", fontsize=10)
fig.savefig(f'fig-08_histogram_integrated_spectral_histograms{nbins}.png')





In [ ]:

diameter = mod.diameter.values

da1 = mod['qv'].values
da2 = mod['qc'].values
da3 = mod['qi+qs'].values

xbins_lim = [np.nanmin(da1), np.nanmax(da1)]
ybins_lim = [np.nanmin(da2), np.nanmax(da2)]
zbins_lim = [np.nanmin(da3), np.nanmax(da3)]
print()
print(f'           xbins_lim   {xbins_lim[0]:10.2e}    {xbins_lim[1]:10.2e}' )
print(f'           ybins_lim   {ybins_lim[0]:10.2e}    {ybins_lim[1]:10.2e}' )
print(f'           zbins_lim   {zbins_lim[0]:10.2e}    {zbins_lim[1]:10.2e}' )

xbins_lim_rounded = [np.floor(np.nanmin(da1)), np.ceil(np.nanmax(da1))]
ybins_lim_rounded = [np.floor(np.nanmin(da2)), np.ceil(np.nanmax(da2))]
zbins_lim_rounded = [np.floor(np.nanmin(da3)), np.ceil(np.nanmax(da3))]
print()
print(f'   xbins_lim_rounded   {xbins_lim_rounded[0]:10.2e}    {xbins_lim_rounded[1]:10.2e}' )
print(f'   ybins_lim_rounded   {ybins_lim_rounded[0]:10.2e}    {ybins_lim_rounded[1]:10.2e}' )
print(f'   zbins_lim_rounded   {zbins_lim_rounded[0]:10.2e}    {zbins_lim_rounded[1]:10.2e}' )

xlim_log = [np.log10(np.nanmin(da1)), np.log10(np.nanmax(da1))]
ylim_log = [np.log10(np.nanmin(da2)), np.log10(np.nanmax(da2))]
zlim_log = [np.log10(np.nanmin(da3)), np.log10(np.nanmax(da3))]
print()
print(f'            xlim_log   {xlim_log[0]:10.2e}    {xlim_log[1]:10.2e}' )
print(f'            ylim_log   {ylim_log[0]:10.2e}    {ylim_log[1]:10.2e}' )
print(f'            zlim_log   {zlim_log[0]:10.2e}    {zlim_log[1]:10.2e}' )

xbins_lim_logrounded = [np.floor(np.log10(np.nanmin(da1))), np.ceil(np.log10(np.nanmax(da1)))]
ybins_lim_logrounded = [np.floor(np.log10(np.nanmin(da2))), np.ceil(np.log10(np.nanmax(da2)))]
zbins_lim_logrounded = [np.floor(np.log10(np.nanmin(da3))), np.ceil(np.log10(np.nanmax(da3)))]
print()
print(f'xbins_lim_logrounded   {xbins_lim_logrounded[0]:10.2e}    {xbins_lim_logrounded[1]:10.2e}' )
print(f'ybins_lim_logrounded   {ybins_lim_logrounded[0]:10.2e}    {ybins_lim_logrounded[1]:10.2e}' )
print(f'zbins_lim_logrounded   {zbins_lim_logrounded[0]:10.2e}    {zbins_lim_logrounded[1]:10.2e}' )


nbins = 512
xbins = np.logspace( *xlim_log, num=nbins , base=10)
ybins = np.logspace( *ylim_log, num=nbins , base=10)
zbins = np.logspace( *zlim_log, num=nbins , base=10)

# histo over all dimensions
# h_sumt   = xhist(mod['qv'],    mod['qc'], bins=[xbins, ybins])
# hw_sumt  = xhist(mod['qv'],    mod['qc'], weights=mod['qi+qs'], bins=[xbins, ybins])
hwf_sumt = xhist(mod['qi+qs'], mod['qc'], weights=mod['qv'],    bins=[zbins, ybins])


# fig, ax = plot_fig8_histo(h_sumt,  figsize=(6, 5), ylim=(1.5, 4.5), xlim=(1.e-7, 2e0), norm=LogNorm(vmin=1e0, vmax=1e5),    clabel="histogram abs. count")
# fig.savefig(f'fig-08_histogram_qv_qc.png')

# fig, ax = plot_fig8_histo(hw_sumt, figsize=(6, 5), ylim=(1.5, 4.5), xlim=(1.e-7, 2e0), norm=LogNorm(vmin=1e-11, vmax=1e-2), clabel=f"histogram weighted by ice water / ({mod['qi'].attrs['units']})")
# fig.savefig(f'fig-08_histogram_timeseries_integrated_qv_qc_qi+qs_nbins{nbins}.png')

fig, ax = plot_fig8_histo(hwf_sumt, figsize=(6, 5), 
                          #ylim=(1.5, 4.5), 
                          #xlim=(1.e-7, 2e0), 
                          norm=LogNorm(),#vmin=1e-11, vmax=1e-2), 
                          clabel=f"histogram weighted by water vapor / ({mod['qi'].attrs['units']})")
fig.savefig(f'fig-08_histogram_timeseries_integrated_qc_qi+qs_qv_nbins{nbins}.png')



In [ ]:
xlim_log = [np.log10(np.nanmin(mod['qv'].values)), np.log10(np.nanmax(mod['qv'].values))]
ylim_log = [np.ceil(np.log10(np.nanmin(mod['qc'].values))), np.ceil(np.log10(np.nanmax(mod['qc'].values)))]

In [ ]:

# histos per time step
cfg['png_path_fig8'] = os.path.join(cfg['png_path'] , f'QL-histogram-qv-qc-qi' ,  )
os.makedirs(cfg['png_path_fig8'], exist_ok=True)

nbins = 128
diameter = mod.diameter.values

xlim_log = [np.log10(np.nanmin(mod['qv'].values)), np.log10(np.nanmax(mod['qv'].values))]
ylim_log = [np.log10(np.nanmin(mod['qc'].values)), np.log10(np.nanmax(mod['qc'].values))]
zbins_lim_log = [np.log10(np.nanmin(mod['qi+qs'].values)), np.log10(np.nanmax(mod['qi+qs'].values))]
print()
print(f'       xlim_log   {xlim_log[0]:10.2e}    {xlim_log[1]:10.2e}' )
print(f'       ylim_log   {ylim_log[0]:10.2e}    {ylim_log[1]:10.2e}' )
print(f'       zbins_lim_log   {zbins_lim_log[0]:10.2e}    {zbins_lim_log[1]:10.2e}' )


xbins = np.logspace( *xlim_log, num=nbins , base=10)
ybins = np.logspace( *ylim_log, num=nbins , base=10)
zbins = np.logspace( *zbins_lim_log, num=nbins , base=10)

plot_frames = False
if plot_frames:
    n_frames = int(mod.time.size)
    frame_file_0 = f'fig-08_histogram_qv_qc_iframe'
    frame_file_1 = f'fig-08_histogram_qv_qc_qi+qs_iframe'
    os.makedirs(os.path.join(cfg['png_path_fig8'], frame_file_0), exist_ok=True)
    os.makedirs(os.path.join(cfg['png_path_fig8'], frame_file_1), exist_ok=True)
    
    for i in tqdm(range(0, n_frames)):
        mo = mod.isel(time=i)
        
        _h  = xhist(mo['qv'], mo['qc'], bins=[xbins, ybins])
        fig, ax = plot_fig8_histo(_h,  figsize=(6, 5), 
                                  ylim=(1.5, 4.5), 
                                  xlim=(1.e-7, 2e0), 
                                  norm=LogNorm(vmin=1e0,vmax=1e3), 
                                  clabel=f"FoO absolut count")
        fig.savefig(os.path.join(cfg['png_path_fig8'], frame_file_0 , f'{i:02d}.png'))
        plt.close(fig)
        
        _hw = xhist(mo['qv'], mo['qc'], weights=mo['qi+qs'], bins=[xbins, ybins])
        fig, ax = plot_fig8_histo(_hw, figsize=(6, 5), 
                                  ylim=(1.5, 4.5), 
                                  xlim=(1.e-7, 2e0), 
                                  norm=LogNorm(vmin=1e-11, vmax=1e-2), 
                                  clabel=f"FoO-weighted-by ice water / ({mod['qi'].attrs['units']})")
        fig.savefig(os.path.join(cfg['png_path_fig8'], frame_file_1 , f'{i:02d}.png'))
        plt.close(fig)
    
    

In [ ]:
if plot_frames:
    in_png  = os.path.join(cfg['png_path_fig8'], frame_file_0,  "%02d.png")
    out_mp4 = os.path.abspath(os.path.join(cfg['png_path_fig8'], frame_file_0[:-7] + f"_{flare_exp_name}_{cfg['resolution']}.mp4"))
    tools.convert_to_video( in_png, out_mp4, resolution="1920:1080", loop_count=2, framerate=15 )
    tools.convert_to_gif(in_png, out_mp4.replace('.mp4', '.gif'), scale_factor=0.5, fps=15)
    # 
    in_png = os.path.join(cfg['png_path_fig8'], frame_file_1, "%02d.png")
    out_mp4 = os.path.abspath(os.path.join(cfg['png_path_fig8'], frame_file_1[:-7] + f"_{flare_exp_name}_{cfg['resolution']}.mp4"))
    tools.convert_to_video( in_png, out_mp4, resolution="1920:1080", loop_count=2, framerate=15 )
    tools.convert_to_gif(in_png, out_mp4.replace('.mp4', '.gif'), scale_factor=0.5, fps=15)
        


# HISTOGRAM STUFF

In [ ]:

# dz = dz.expand_dims(dim='diameter')#
# dz = mod['dz'].assign_coords({'diameter': mod.diameter})
# dz2
# (mod.diameter_edges.isel(diameter_edges=slice(1, None))-mod.diameter_edges.isel(diameter_edges=slice(None, None))).shape




In [ ]:
mod.diameter_edges

In [ ]:

d_z = mod.dz.values
d_Vol  = cfg['delta_x'] * cfg['delta_y'] * d_z
mo_Vol_dict = []
mo_Vol_max_list = []
for ibin in range(0, mod.diameter.size):
    delta_D = mod.diameter_edges[ibin+1] - mod.diameter_edges[ibin]
    mo_Vol = mod[['qfw', 'qw']].isel(diameter=ibin) 
    mo_Vol = mo_Vol * d_Vol * delta_D
    mo_Vol_dict.append( mo_Vol.expand_dims('diameter', axis=-1) )


mo_Vol_spec = xr.concat(mo_Vol_dict, dim='diameter', )
mo_Vol_spec = xr.where(mo_Vol_spec>0, mo_Vol_spec, np.nan)

with ProgressBar():
    mo_Vol_spec = mo_Vol_spec.persist()


In [ ]:

with ProgressBar():
    sum_h = sum_h.persist()


In [ ]:
print(np.nanmin(sum_h), np.nanmax(sum_h))
xlim = (xbins.min(), xbins.max())
ylim = (ybins.min(), ybins.max())
vlim = (10**np.ceil(np.log10(sum_h.min().values)), 10**(np.floor(np.log10(sum_h.max().values))+1))
cb_kw = {"cbar_kwargs": {"orientation": "vertical", "location": "right", "extend": "both", "shrink": 0.9, "pad":0.1 , 'label': "histogram / (# per bin)"}}

fig, ax = setup_histo_fig(figsize=(6,5), nrows=1, grid=False, labelsize=10)

sum_h.plot(ax=ax, 
           yscale='log', 
           xscale='log',
           robust=True, 
           norm=LogNorm(vmin=vlim[0], vmax=vlim[1])  ,
           alpha=0.9, 
           xlim=xlim, 
           ylim=ylim,
           cmap=new_jet4,
           cbar_kwargs={"orientation": "vertical", 
                       "location": "right", 
                       "extend": "both", 
                       "shrink": 0.9, "pad":0.025,
                       'label': "FoO liq/frozen water \nspectra integral", })
ax.plot([1e-18, 1e7], [1e-18, 1e7], '--')
ax.set_xlim(1e-18, 1e7)
ax.set_ylim(1e-18, 1e7)
ax.set_ylabel(f"liquid water spectra / ({mod['qw'].attrs['units']})", fontsize=10)
ax.set_xlabel(f"frozen water spectra / ({mod['qfw'].attrs['units']})", fontsize=10)
fig.savefig(f'fig-08_histogram_integrated_spectral_histograms{nbins}.png')




In [ ]:
if plot_frames:
    print(np.nanmin(sum_h), np.nanmax(sum_h))
    xlim = (xbins.min(), xbins.max())
    ylim = (ybins.min(), ybins.max())
    vlim = (10**np.ceil(np.log10(sum_h.min().values)), 10**(np.floor(np.log10(sum_h.max().values))+1))
    vlim = (1e0, 1e4)
    cb_kw = {"cbar_kwargs": {"orientation": "vertical", "location": "right", "extend": "both", "shrink": 0.9, "pad":0.1 , 'label': "histogram / (# per bin)"}}
    cfg['png_path_fig9'] = './QL-qw-qfw-histo/'
    diameter_µm = mod.diameter_edges.values
    os.makedirs(cfg['png_path_fig9'] + 'pngs', exist_ok=True)
    start_frame=15
    for ibin in tqdm(range(start_frame, ds_hbin.diameter.size-15)):
        fig, ax = setup_histo_fig(figsize=(6,5), nrows=1, grid=False, labelsize=10)
        _h = ds_hbin.isel(diameter=ibin)
        _h.plot(ax=ax, 
                   yscale='log', 
                   xscale='log',
                   robust=True, 
                   norm=LogNorm(vmin=vlim[0], vmax=vlim[1])  ,
                   alpha=0.9, 
                   xlim=xlim, 
                   ylim=ylim,
                   cmap=new_jet4,
                   cbar_kwargs={"orientation": "vertical", 
                               "location": "right", 
                               "extend": "both", 
                               "shrink": 0.9, "pad":0.025,
                               'label': "FoO liq/frozen water \nspectra integral", })
        ax.plot([1e-18, 1e7], [1e-18, 1e7], '--')
        ax.set_xlim(1e-18, 1e7)
        ax.set_ylim(1e-18, 1e7)
        ax.set_ylabel(f"liquid water spectra / ({mod['qw'].attrs['units']})", fontsize=10)
        ax.set_xlabel(f"frozen water spectra / ({mod['qfw'].attrs['units']})", fontsize=10)
        ax.set_title(f'bin({ibin:02d})  {diameter_µm[ibin]:5.1f} -- {diameter_µm[ibin+1]:5.1f} µm')
        fig.savefig(cfg['png_path_fig9'] + f'pngs/fig-09_histogram_integrated_spectral_histograms{nbins}_{ibin-start_frame:02d}.png')
        plt.close(fig)
    
    


In [ ]:
# Create video and gif
if plot_frames:
    input_pattern = os.path.join(cfg['png_path_fig9'], f'pngs/fig-09_histogram_integrated_spectral_histograms{nbins}_%2d.png')
    output_mp4 = os.path.join(cfg['png_path_fig9'], f"9-eriswil_histo_qw-qfw_{flare_exp_name}_{cfg['resolution']}.mp4")
    output_gif = output_mp4.replace('.mp4', '.gif')
    
    tools.convert_to_video(input_pattern, output_mp4, resolution="1920:1080", loop_count=2, framerate=5, )
    tools.convert_to_gif(input_pattern, output_gif, scale_factor=0.5, fps=5)
    
    print(f"Created: {output_mp4}")
    print(f"Created: {output_gif}")

In [ ]:

# with ProgressBar():
#     sum_hw = xr.where(sum_hw>0, sum_hw, np.nan).persist()
# # xlim = (np.log10(xlim_log[0]), np.log10(xlim_log[1]))
# # ylim = (np.log10(ylim_log[0]), np.log10(ylim_log[1]))
# print(np.nanmin(sum_hw), np.nanmax(sum_hw))
# xlim = (xbins.min(), xbins.max())
# ylim = (ybins.min(), ybins.max())
# vlim = (10**np.ceil(np.log10(sum_hw.min().values)), 10**np.floor(np.log10(sum_hw.max().values)))
# cb_kw = {"cbar_kwargs": {"orientation": "vertical", "location": "right", "extend": "both", "shrink": 0.9, "pad":0.1 , 'label': "histogram / (# per bin)"}}

# fig, ax = setup_histo_fig(figsize=(6,5), nrows=1, grid=False, labelsize=10)

# sum_hw.plot(ax=ax, 
#            yscale='log', xscale='log',
#            robust=True, 
#            norm=LogNorm(vmin=vlim[0], vmax=vlim[1])  ,
#            alpha=0.9, 
#            xlim=xlim, 
#            ylim=ylim,
#            cmap=new_jet4,
#            cbar_kwargs={"orientation": "vertical", 
#                        "location": "right", 
#                        "extend": "both", 
#                        "shrink": 0.9, "pad":0.025,
#                        'label': "FoO liq/frozen water \nspectra integral", })

# ax.set_ylabel(f"water vapor / ({mod['qv'].attrs['units']})", fontsize=10)
# ax.set_xlabel(f"liquid water / ({mod['qc'].attrs['units']})", fontsize=10)
# fig.savefig(f'fig-08_histogram_integrated_spectral_histograms_weighted.png')




In [ ]:
# xlim = (np.log10(xlim_log[0]), np.log10(xlim_log[1]))
# ylim = (np.log10(ylim_log[0]), np.log10(ylim_log[1]))
print(np.nanmin(sum_h), np.nanmax(sum_h))
xlim = (xbins.min(), xbins.max())
ylim = (ybins.min(), ybins.max())
vlim = (10**np.ceil(np.log10(sum_h.min().values)), 10**np.floor(np.log10(sum_h.max().values)))
cb_kw = {"cbar_kwargs": {"orientation": "vertical", "location": "right", "extend": "both", "shrink": 0.9, "pad":0.1 , 'label': "histogram / (# per bin)"}}

fig, axes = setup_histo_fig(figsize=(6,8), nrows=2, grid=False, labelsize=10)

sum_h.plot(ax=axes[0], 
           yscale='log', xscale='log',
           robust=True, 
           norm=LogNorm(vmin=vlim[0], vmax=vlim[1])  ,
           alpha=0.9, 
           xlim=xlim, 
           ylim=ylim,
           cmap=new_jet4,
           cbar_kwargs={"orientation": "vertical", 
                       "location": "right", 
                       "extend": "both", 
                       "shrink": 0.9, "pad":0.025,
                       'label': "FoO liq/frozen water \nspectra integral", })

sum_hw.plot(ax=axes[1], 
           yscale='log', xscale='log',
           robust=True, 
           norm=LogNorm(vmin=vlim[0], vmax=vlim[1])  ,
           alpha=0.9, 
           xlim=xlim, 
           ylim=ylim,
           cmap=new_jet4,
           cbar_kwargs={"orientation": "vertical", 
                       "location": "right", 
                       "extend": "both", 
                       "shrink": 0.9, "pad":0.025,
                       'label': "FoO liq/frozen water \nspectra integral", })

ax[0].set_ylabel(f"water vapor / ({mod['qv'].attrs['units']})", fontsize=10)
ax[1].set_ylabel(f"water vapor / ({mod['qv'].attrs['units']})", fontsize=10)
ax[1].set_xlabel(f"liquid water / ({mod['qc'].attrs['units']})", fontsize=10)
fig.savefig(f'fig-08_histogram_integrated_spectral_histograms.png')




In [ ]:
# fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(3, 2))
# h = hist_list
# cb_kw = {"cbar_kwargs": {"orientation": "vertical", "location": "right", "extend": "both", "shrink": 0.9, "pad":0.1 }}
# xr.where(h>0, h, np.nan).plot(ax=ax[0], xscale='log', yscale='log', robust=True, norm=LogNorm(), alpha=0.9, cmap=cmap_sv_fade, **cb_kw)
# for h, iax in zip(hist_list, ax.flatten()):
#     xr.where(h>0, h, np.nan).plot(ax=iax, xscale='log', yscale='log', robust=True, cmap=cmap_sv_fade, **cb_kw) #LogNorm(vmin=1e1, vmax=5e4)


In [ ]:
def setup_histo_fig2(nrows=2, ncols=2, figsize=(5, 8)):
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, sharex=True, sharey=True)
    # Configure time series axis
    for ax in axes:
        set_name_tick_params(ax)
        # Configure histo axis
        ax.tick_params(axis='both', which='major', labelsize=cfg['tick_size']-2, direction='in', 
                       top=True, right=True, bottom=True, left=True)
        ax.tick_params(axis='both', which='minor', labelsize=cfg['tick_size']-4, direction='in', 
                       top=True, right=True, bottom=True, left=True)
        # ax.set_xlabel('diameter', fontsize=cfg['tick_size']-1)
        # ax.set_ylabel('cumulative\ncount', fontsize=cfg['tick_size']-1)
        ax.patch.set_alpha(0.8)
        ax.tick_params(which='major', length=5)
        ax.tick_params(which='minor', length=3)
        ax.xaxis.set_ticks_position('both')
        ax.yaxis.set_ticks_position('both')
        # ax.yaxis.tick_right()
        # 
        # ax.xaxis.set_label_position("top")
        # ax.yaxis.set_label_position("right")
        # ax.set_xscale('log')
        # ax.set_yscale('log')
    return fig, axes

In [ ]:
cb_kw = {"cbar_kwargs": {"orientation": "vertical", "location": "right", "extend": "both", "shrink": 0.9, "pad":0.1 , 'label': "histogram / (# per bin)"}}
diameter = mod.diameter.values[:]
ncols = 5
nrows = 6
hist_list = hwbin_list
fig, axes = setup_histo_fig2(ncols=ncols, nrows=nrows, figsize=(ncols*3.5, nrows*2.3))
for i, (h, iax) in enumerate(zip(hist_list[24:-8], axes.flatten())):
    xr.where(h>0, h, np.nan).plot(ax=iax, xscale='log', yscale='log', robust=True, cmap=new_jet2, norm=LogNorm(vmin=1e0, vmax=1e4), **cb_kw) 
    iax.set_title(f'{diameter[i]:.1f} -- {diameter[i+1]:.1e} µm')
    iax.set(xlabel='', ylabel='') if (i < axes.shape[0] - 1) else iax.set(xlabel='frozen water / (gL$_{-1}$)', ylabel='liquid water / (gm$_{-3}$)')
    iax.grid(True, which='major', linestyle='--', linewidth='0.1', color='black', alpha=0.55, zorder=99.1)

fig.savefig('fig-7_qw-qwf_wei_qs+qi.png')

In [ ]:
os.makedirs(os.path.join(cfg['png_path'], '../QL-histo'), exist_ok=True)
cb_kw = {"cbar_kwargs": {"orientation": "vertical", "location": "right", "extend": "both", "shrink": 0.9, "pad":0.1 , 'label': "histogram / (# per bin)"}}
diameter = mod.diameter.values[24:-7]
ncols=1
nrows=1
for i, h  in enumerate(hist_list[24:-8]):
    
    fig, ax = setup_histo_fig2(ncols=ncols, nrows=2, figsize=(ncols*6, nrows*5))
    # fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(5, 4), constrained_layout=True, sharex=True, sharey=True)
    xr.where(h>0, h, np.nan).plot(ax=ax[0], xscale='log', yscale='log', robust=True, cmap=new_jet4, norm=LogNorm(vmin=1e0, vmax=1e3), **cb_kw) 
    ax[0].set_title(f'{diameter[i]:.1f} -- {diameter[i+1]:.1f} µm')
    ax[0].set(xlabel='frozen water / (gL$^{-1}$)', ylabel='liquid water / (gm$^{-3}$)')

        xr.where(h>0, h, np.nan).plot(ax=ax[0], xscale='log', yscale='log', robust=True, cmap=new_jet4, norm=LogNorm(vmin=1e0, vmax=1e3), **cb_kw) 
    ax[0].set_title(f'{diameter[i]:.1f} -- {diameter[i+1]:.1f} µm')
    ax[0].set(xlabel='frozen water / (gL$^{-1}$)', ylabel='liquid water / (gm$^{-3}$)')

    xr.where(h_>0, h, np.nan).plot(ax=ax[0], xscale='log', yscale='log', robust=True, cmap=new_jet4, norm=LogNorm(vmin=1e0, vmax=1e3), **cb_kw) 
    ax[0].set_title(f'{diameter[i]:.1f} -- {diameter[i+1]:.1f} µm')
    ax[0].set(xlabel='frozen water / (gL$^{-1}$)', ylabel='liquid water / (gm$^{-3}$)')

        xr.where(h>0, h, np.nan).plot(ax=ax[0], xscale='log', yscale='log', robust=True, cmap=new_jet4, norm=LogNorm(vmin=1e0, vmax=1e3), **cb_kw) 
    ax[0].set_title(f'{diameter[i]:.1f} -- {diameter[i+1]:.1f} µm')
    ax[0].set(xlabel='frozen water / (gL$^{-1}$)', ylabel='liquid water / (gm$^{-3}$)')
    fig.savefig(cfg['png_path'] + f'../QL-histo/{str(i).zfill(2)}_qw-qfw.png', bbox_inches='tight', dpi=300)
    plt.close(fig)
    

In [ ]:
# Create video and gif
input_pattern = os.path.join(cfg['png_path'], f"../QL-histo/%02d_qw-qfw.png")
output_mp4 = os.path.join(cfg['png_path'], f"../6-eriswil_qw-qfw_{flare_exp_name}_{cfg['resolution']}.mp4")
output_gif = output_mp4.replace('.mp4', '.gif')

tools.convert_to_video(input_pattern, output_mp4, resolution="1920:1080", loop_count=2, framerate=5)
tools.convert_to_gif(input_pattern, output_gif, scale_factor=0.5, fps=5)

print(f"Created: {output_mp4}")
print(f"Created: {output_gif}")


In [ ]:
from xhistogram.xarray import histogram as xhist
nbins = 512
mod.diameter.values
fig, ax = setup_histo_fig()
# xbins = np.linspace( np.nanmin(masked_qv).round(), np.ceil(np.nanmax(masked_qv)), nbins )
# ybins = np.linspace( np.nanmin(masked_qc).round(), np.ceil(np.nanmax(masked_qc)), nbins )
xbins = np.logspace( -1, 0.7, num=nbins , base=10)
ybins = np.logspace( -7, 0, num=nbins , base=10)
bins = [xbins, ybins]
hist_list = []
for ibin in range(30, mod.diameter.size):
    mo = mod.isel(diameter=ibin)
    hist_list.append(xhist(mo['qw'], mo['qfw'], bins=bins))
    #hw = xhist(mod['qv'], mod['qc'], weights=mod['qi']+mod['qs'], bins=bins)
print(len(hist_list))
# hist_list[100]

In [ ]:
ncols = 2
nrows = 2#mod.diameter.size//ncols
fig, ax = plt.subplots(ncols=8, nrows=nrows, figsize=(ncols*3, nrows*2))
for ii, iax in enumerate(ax.flatten()):
    h = hist_list[ii]
    xr.where(h>0, h, np.nan).plot(ax=iax, xscale='log', yscale='log', norm=LogNorm(vmin=1e1, vmax=5e4), cmap=new_jet2, add_colorbar=False) #LogNorm(vmin=1e1, vmax=5e4)


In [ ]:
hhhhh= []
for ii in range(len(hist_list)):
    hhhhh.append(xr.where(hist_list[ii]>0, hist_list[ii], np.nan))
hhhhh = np.array(hhhhh)

In [ ]:
np.nanmin(hhhhh), np.nanmax(hhhhh)

## 200x160 comparison figures (joint, diameter-resolved, threshold occurrence)

In [ ]:
from xhistogram.xarray import histogram as xhist


def _load_run_mod(cs_run, ep_dom, cs_run_idx, model_data_root, tracking_by_var='qi+qs'):
    model_data_path = model_data_root + f'/RUN_ERISWILL_{ep_dom}x100/ensemble_output/{cs_run}/'
    extpar_file = model_data_root + f'/RUN_ERISWILL_{ep_dom}x100/COS_in/extPar_Eriswil_{ep_dom}.nc'
    json_file = glob.glob(model_data_path + '*.json')[0]

    with open(json_file, 'r') as f:
        meta = json.load(f)

    flist_3d = sorted(glob.glob(model_data_path + '3D_??????????????.nc'))
    exp_names = [f.split('/')[-1].split('_')[-1].split('.')[0] for f in flist_3d]
    flare_exp_name = [exp for exp in exp_names if meta[exp]['INPUT_ORG']['sbm_par']['lflare']][cs_run_idx]
    flare_exp_nc_file = [f for f in flist_3d if flare_exp_name in f][0]

    reduced_domain = dict(latitude=slice(None, 47.07425 + 2.0 * 0.001), longitude=slice(None, 7.90522 + 2.0 * 0.001))

    ds_3d = fetch_3d_data(
        flare_exp_nc_file,
        extpar_file,
        meta[flare_exp_name]['INPUT_ORG'],
        var_sets=['meteo', 'spec'],
        chunks={'time': 1, 'altitude': 1},
    )
    ds_3d = update_dataset_metadata(ds_3d)
    ds_3d = ds_3d.isel(altitude=slice(80, None)).sel(reduced_domain)
    ds_3d = convert_units_3d(ds_3d, ds_3d['rho'])

    mod = ds_3d[['qv', 'qc', 'qi', 'qs', 'qfw', 'qw', 'nf', 'nw']]
    d_um = define_bin_boundaries() * 1.0e6 * 2.0
    mod = mod.assign_coords({'diameter_edges': xr.DataArray(d_um, dims='diameter_edges')})
    mod[tracking_by_var] = xr.where(mod['qi'] + mod['qs'] > 0, mod['qi'] + mod['qs'], np.nan)
    mod[tracking_by_var].attrs['units'] = mod['qi'].attrs.get('units', '')

    return mod, flare_exp_name


def _safe_log_bins(da, nbins=96):
    values = np.asarray(da.where(np.isfinite(da) & (da > 0)).values).ravel()
    values = values[np.isfinite(values)]
    vmin, vmax = np.nanpercentile(values, [0.1, 99.9])
    vmin = max(vmin, np.finfo(float).tiny)
    return np.logspace(np.log10(vmin), np.log10(vmax), nbins)


def _compute_joint_freq(mod, xbins, ybins):
    h = xhist(mod['qv'], mod['qc'], bins=[xbins, ybins])
    return h / h.sum()


def _compute_diameter_resolved_qfreq(mod, xbins, ybins):
    rows = []
    for ibin in range(mod.diameter.size):
        mo = mod.isel(diameter=ibin)
        h = xhist(mo['qfw'], mo['qw'], bins=[xbins, ybins]).sum('qw_bin').expand_dims('diameter')
        rows.append(h)
    ds = xr.concat(rows, dim='diameter').assign_coords({'diameter': mod.diameter})
    return ds / ds.sum('qfw_bin')


def _compute_exceedance(mod, var, threshold):
    return xr.where(mod[var] >= threshold, 1.0, 0.0).mean(dim=('latitude', 'longitude'))


runs_200x160 = [row for row in cs_runs if row[1] == '200x160'][:2]
assert len(runs_200x160) == 2, 'Need exactly two 200x160 runs for comparison figures.'

run_data = []
for cs_run, ep_dom, cs_run_idx in runs_200x160:
    mod_i, flare_name_i = _load_run_mod(cs_run, ep_dom, cs_run_idx, model_data_root=model_data_root, tracking_by_var='qi+qs')
    run_data.append({'run_id': cs_run, 'flare_exp_name': flare_name_i, 'mod': mod_i})

mod1, mod2 = run_data[0]['mod'], run_data[1]['mod']
label1, label2 = run_data[0]['run_id'], run_data[1]['run_id']

qv_bins = _safe_log_bins(xr.concat([mod1['qv'], mod2['qv']], dim='stack_qv'))
qc_bins = _safe_log_bins(xr.concat([mod1['qc'], mod2['qc']], dim='stack_qc'))
qfw_bins = _safe_log_bins(xr.concat([mod1['qfw'], mod2['qfw']], dim='stack_qfw'))
qw_bins = _safe_log_bins(xr.concat([mod1['qw'], mod2['qw']], dim='stack_qw'))

joint1 = _compute_joint_freq(mod1, qv_bins, qc_bins)
joint2 = _compute_joint_freq(mod2, qv_bins, qc_bins)
joint_diff = joint2 - joint1

# Figure A: joint frequency qv vs qc + run difference
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
vmax_joint = float(np.nanmax([joint1.max().values, joint2.max().values]))
vmin_joint = max(vmax_joint * 1e-5, np.finfo(float).tiny)
for i, (da, ttl) in enumerate([(joint1, label1), (joint2, label2)]):
    xr.where(da > 0, da, np.nan).plot(
        ax=ax[i],
        xscale='log',
        yscale='log',
        cmap='viridis',
        norm=LogNorm(vmin=vmin_joint, vmax=vmax_joint),
        cbar_kwargs={'label': 'joint frequency'},
    )
    ax[i].set_title(f'Joint freq qv/qc: {ttl}')
    ax[i].set_xlabel('qv')
    ax[i].set_ylabel('qc')

vd = float(np.nanmax(np.abs(joint_diff.values)))
joint_diff.plot(
    ax=ax[2],
    xscale='log',
    yscale='log',
    cmap='coolwarm',
    vmin=-vd,
    vmax=vd,
    cbar_kwargs={'label': f'freq diff ({label2} - {label1})'},
)
ax[2].set_title('Joint freq difference')
ax[2].set_xlabel('qv')
ax[2].set_ylabel('qc')
fig.savefig('fig-comparison-a-joint-freq-200x160.png', dpi=300)


# Figure B: diameter-resolved qfw/qw histogram summary + run difference
diam_q1 = _compute_diameter_resolved_qfreq(mod1, qfw_bins, qw_bins)
diam_q2 = _compute_diameter_resolved_qfreq(mod2, qfw_bins, qw_bins)
diam_qdiff = diam_q2 - diam_q1

fig, ax = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
vmax_d = float(np.nanmax([diam_q1.max().values, diam_q2.max().values]))
vmin_d = max(vmax_d * 1e-5, np.finfo(float).tiny)
for i, (da, ttl) in enumerate([(diam_q1, label1), (diam_q2, label2)]):
    xr.where(da > 0, da, np.nan).plot(
        ax=ax[i],
        x='diameter',
        y='qfw_bin',
        xscale='log',
        yscale='log',
        cmap='magma',
        norm=LogNorm(vmin=vmin_d, vmax=vmax_d),
        cbar_kwargs={'label': 'freq(qfw | diameter)'},
    )
    ax[i].set_title(f'Diameter-resolved qfw/qw: {ttl}')
    ax[i].set_xlabel('diameter (um)')
    ax[i].set_ylabel('qfw')

vdd = float(np.nanmax(np.abs(diam_qdiff.values)))
diam_qdiff.plot(
    ax=ax[2],
    x='diameter',
    y='qfw_bin',
    xscale='log',
    yscale='log',
    cmap='coolwarm',
    vmin=-vdd,
    vmax=vdd,
    cbar_kwargs={'label': f'freq diff ({label2} - {label1})'},
)
ax[2].set_title('Diameter-resolved difference')
ax[2].set_xlabel('diameter (um)')
ax[2].set_ylabel('qfw')
fig.savefig('fig-comparison-b-diameter-resolved-hist-200x160.png', dpi=300)


# Figure C: threshold occurrence/exceedance (time-height) + run difference
thresholds = {'qi+qs': 1e-8, 'qc': 1e-8}
occ = {}
for var, th in thresholds.items():
    occ[(label1, var)] = _compute_exceedance(mod1, var, th)
    occ[(label2, var)] = _compute_exceedance(mod2, var, th)

fig, ax = plt.subplots(2, 3, figsize=(15, 7.2), constrained_layout=True, sharex=True, sharey=True)
for r, var in enumerate(['qi+qs', 'qc']):
    da1 = occ[(label1, var)]
    da2 = occ[(label2, var)]
    dad = da2 - da1

    for c, (da, ttl, cmap) in enumerate(
        [
            (da1, f'{label1}: {var} >= {thresholds[var]:.1e}', 'viridis'),
            (da2, f'{label2}: {var} >= {thresholds[var]:.1e}', 'viridis'),
            (dad, f'diff ({label2} - {label1})', 'coolwarm'),
        ]
    ):
        kw = dict(x='time', y='altitude', cmap=cmap, add_colorbar=True)
        if c < 2:
            kw['vmin'] = 0.0
            kw['vmax'] = 1.0
            kw['cbar_kwargs'] = {'label': 'occurrence frequency'}
        else:
            vm = float(np.nanmax(np.abs(dad.values)))
            kw['vmin'] = -vm
            kw['vmax'] = vm
            kw['cbar_kwargs'] = {'label': 'frequency difference'}
        da.plot(ax=ax[r, c], **kw)
        ax[r, c].set_title(ttl)
        ax[r, c].set_xlabel('time')
        ax[r, c].set_ylabel('altitude')

fig.savefig('fig-comparison-c-threshold-occurrence-200x160.png', dpi=300)

print('Saved: fig-comparison-a-joint-freq-200x160.png')
print('Saved: fig-comparison-b-diameter-resolved-hist-200x160.png')
print('Saved: fig-comparison-c-threshold-occurrence-200x160.png')